# 03, the matcher without schemes

The first matcher compared versions as text. As text, 4.10.0 comes before 4.9.0, so a component above a fix looked affected and a component below a limit looked safe. This notebook runs both comparisons over every single interval range the committed exposure matched, and lists the findings the text comparison would have lost.

In [1]:
import json
import re
from pathlib import Path

from univers.version_range import VersionRange

findings = []
for name in ("findings_pin.json", "findings_today.json"):
    data = json.loads(Path(f"results/exposure/{name}").read_text())
    for block in data["products"].values():
        findings.extend(block["findings"])
pairs = {
    (m["matched_range"], f["version"])
    for f in findings
    for m in f["matches"]
    if (m.get("matched_range") or "").startswith("vers:")
}
print(len(findings), "findings,", len(pairs), "distinct range and version pairs")

1275 findings, 676 distinct range and version pairs


In [2]:
OPS = re.compile(r"^(>=|<=|<|>)(.*)$")


def simple(vers: str) -> list[tuple[str, str]] | None:
    # One interval at most: a lower bound, an upper bound or both. A range of several intervals
    # needs the vers ordering rules on top of the comparison, so it is left out of this check.
    parts = [OPS.match(x) for x in vers.split("/", 1)[1].split("|")]
    if len(parts) > 2 or any(m is None for m in parts):
        return None
    ops = [m.group(1) for m in parts]
    if len(ops) == 2 and not (ops[0].startswith(">") and ops[1].startswith("<")):
        return None
    return [(m.group(1), m.group(2)) for m in parts]


def as_text(constraints: list[tuple[str, str]], version: str) -> bool:
    # The first matcher: the same interval, every bound compared as plain text.
    test = {"<": str.__lt__, "<=": str.__le__, ">": str.__gt__, ">=": str.__ge__}
    return all(test[op](version, bound) for op, bound in constraints)


def by_scheme(vers: str, version: str) -> bool:
    r = VersionRange.from_string(vers)
    return r.version_class(version) in r


rows = []
for v, ver in sorted(pairs):
    c = simple(v)
    if c is None:
        continue
    rows.append((v, ver, as_text(c, ver), by_scheme(v, ver)))
missed = [r for r in rows if r[3] and not r[2]]
print(f"{len(rows)} pairs with a single interval (of {len(pairs)}); text comparison misses {len(missed)}")
for v, ver, _, _ in missed[:8]:
    print(f"  {ver} against {v}")

626 pairs with a single interval (of 676); text comparison misses 124
  0.17.8 against vers:cargo/<0.17.12
  0.17.8 against vers:cargo/>=0.0.0-0|<0.17.12
  1.0.86 against vers:cargo/>=0.0.0-0|<1.0.103
  0.23.10 against vers:cargo/>=0.23.9|<0.23.46
  0.3.36 against vers:cargo/>=0.3.6|<0.3.47
  0.10.0 against vers:cargo/>=0.8.0
  0.11.0 against vers:cargo/>=0.8.0
  0.10.0 against vers:cargo/>=0.8.0|<=0.11.0


Every pair above is a finding the scheme aware matcher made that the text comparison would have dropped, because as text 0.17.8 sorts after 0.17.12 and 0.10.0 before 0.8.0.

What changed in the build: packages/match/src/unscheduled_match/ranges.py parses each version with the class of its ecosystem (univers), and the exposure step runs osv-scanner on the same SBOMs, so a disagreement between the two cannot pass unexplained (results/exposure/disagreements.json).